# Readme E-Rara

Dieses Jupyter Notebook bereitet Datenobjekte und Metadaten aus dem E-Rara-Bestand vor für die digitale Langzeitarchivierung (DLZA) der ZHB Luzern, basierend auf der GOCFL implementierung von Jürgen Enge, https://github.com/je4/gocfl .



## Basiskonfiguration


Die E-Rara-Signaturen setzen sich aus dem E-Rara DOI zusammen. Im Config-File wird die Abteilung hinzugefügt, das Script ergänzt den DOI. 


### Config.py

Beispieldaten für ZHB E-Rara. Anpassungen können in der config.py vorgenommen werden. 

    address = 'mailto:someone@internet.com'
    collection = 'ZHB E-Rara'
    collection_id = 'zhb_erara'
    last_changed = 'yyyy-mm-dd'
    organisation = 'Zentral- und Hochschulbibliothek Luzern'
    organisation_id = 'zhb'
    signature = 'zhb_'
    
### OAI configuration 

Für die E-Rara-Bestände wird die Zenodo-OAI-Schnittstelle verwendet, da hier die Original-ZIP-Kapseln liegen und auf alle andern Systeme verlinkt wird (Alma, E-Rara).

Zenodo: siehe https://developers.zenodo.org/#oai-pmh

set name: lara_e-rara
listrecords: user-lara_e-rara

### Export

Für jeden einzelnen record wird eine info.json-Datei erstellt im Format info/signature.json.
Das ganze Set wird am Ende noch als json- und Excel-Datei exportiert ins directory 'fulldum' als menschenlesbarer Nachweis, welche Datenobjekte eingelagert wurden. 

### Marcxml aus Alma (SRU)

Mit der alma_id werden die MARC-Daten via SRU aus Alma extrahiert und abgespeichert unter metadata/signature.xml


### Ablage Datenobjekte 
Die E-Rara-Zipkapseln der ZHB sind nach folgender Struktur benannt:
DOI (Punkt/Schrägstrich ersetzt durch Unterstrich) _ (Datum/Workflow?) _ master _ version . zip

Beispiel:
10_3931_e-rara-86416_20201027T114533_master_ver1.zip

Hier wird vorausgesetzt, dass die Daten mit dieser Bezeichnung im Ordner objects liegen.


### TODO Create Befehl für gocfl generieren

Die gocfl create Befehle für die E-Rara-Sammlung (alle Objekte) werden gemäss https://github.com/je4/gocfl/blob/main/docs/create.md erstellt mit der Signature. 

Muster:
gocfl create P:/temp/archiv P:/temp/testdata/zhb_erara_86774/object metadata:p:/temp/testdata/zhb_erara_86774/metadata --config p:/temp/config/gocfl.toml -i "zhb_erara_86774" --ext-NNNN-metafile-source p:/temp/testdata/zhb_erara_86774.json 


In [1]:
from sickle import Sickle
import json
import config
import xml.etree.ElementTree as ET
import requests
import pandas as pd


# Initialize the client by passing the base URL and fetch records from the OAI Set

base_url = 'https://zenodo.org/oai2d'
prefix = 'oai_dc'
set_name = 'user-lara_e-rara'

sickle = Sickle(base_url)
records = sickle.ListRecords(metadataPrefix=prefix, set=set_name)
record = records.next()

completed_iterating = False
recordCount = 1
completeSet = []

# Iterate through records and collect metadata info for json export
# Uncomment one of the next 2 lines for testing/production:

#while not completed_iterating:
while recordCount < 5:
    try:
        
        infoSet = {}
                
        infoSet["address"] = config.address
        infoSet["collection"] = config.collection
        infoSet["collection_id"] = config.collection_id
        infoSet["created"] = record.metadata["date"][0]
        infoSet["description"] = record.metadata["description"][0]
        infoSet["identifiers"] = record.metadata["identifier"]
        infoSet["ingest_workflow"] = config.ingest_workflow
        infoSet["keywords"] = config.keywords
        infoSet["last_changed"] = config.last_changed
        infoSet["organisation"] = config.organisation
        infoSet["organisation_id"] = config.organisation_id
        infoSet["references"] = record.metadata["relation"]
        
        # add e-rara doi to identifiers:
        erara_doi = infoSet["references"][0][4:]  #        print(erara_doi)
        infoSet["identifiers"].insert(0, erara_doi)
        erara_signature = erara_doi.replace('.','_').replace('/','_') #         print(erara_signature)
        
        # add alma_id to identifiers:
        alma_link = infoSet["references"][1]
        alma_id = alma_link.partition('alma')[2]
        infoSet["identifiers"].insert(0, str(alma_id))         #print(infoSet["identifiers"])        
        
        infoSet["sets"] = config.sets
        signature = config.signature+erara_signature
        infoSet["signature"] = signature
        print(signature)
        infoSet["title"] = record.metadata["title"][0]
        infoSet["user"] = record.metadata["creator"][0]
        #debugging: print(infoSet)
        
        # prepare filename for json export
        info_json = json.dumps(infoSet, indent=4, ensure_ascii=False)
        infofile = f"info/{signature}.json"
        with open(infofile, "w") as outfile:
            outfile.write(info_json)
            print(f"info.json saved as {infofile}")
        
        # add info to completeSet
        completeSet.append(infoSet)
       
        # get metadata from Alma OAI as MARCXML
        
        sru_url = "https://slsp-rzs.alma.exlibrisgroup.com/view/sru/41SLSP_RZS"
        query = f"{sru_url}?version=1.2&operation=searchRetrieve&recordSchema=marcxml&query=rec.id={alma_id}"
        response = requests.get(query)
        if response.status_code != 200:
            raise Exception(f"SRU request failed with status code {response.status_code}")

        # Save the response content (MARCXML) to a file
        metafile = f"metadata/{signature}.xml"
        with open(metafile, 'wb') as file:
            file.write(response.content)
            print(f"Record with ID {alma_id} saved as {metafile}")    

        #continue with next record
        recordCount = recordCount +1
        record = records.next()
        
    except StopIteration:
        completed_iterating = True

# Writing completeSet as json file
fulldump = json.dumps(completeSet, indent=4, ensure_ascii=False)
fulljsonfile = "fulldump/erara_complete_set.json"
with open(fulljsonfile, "w") as outfile:
    outfile.write(fulldump)
    print(f"---\nfulldump written to {fulljsonfile}")
    
# Writing completeSet as Excel file
fullexcelfile = "fulldump/erara_complete_set.xlsx"
df_json = pd.read_json(fulljsonfile)
df_json.to_excel(fullexcelfile)
print(f"Saved fulldump in excel file as {fullexcelfile}")


zhb_10_3931_e-rara-86409
info.json saved as info/zhb_10_3931_e-rara-86409.json
Record with ID 997037850105505 saved as metadata/zhb_10_3931_e-rara-86409.xml
zhb_10_3931_e-rara-87374
info.json saved as info/zhb_10_3931_e-rara-87374.json
Record with ID 994403000105505 saved as metadata/zhb_10_3931_e-rara-87374.xml
zhb_10_3931_e-rara-86372
info.json saved as info/zhb_10_3931_e-rara-86372.json
Record with ID 992994940105505 saved as metadata/zhb_10_3931_e-rara-86372.xml
zhb_10_3931_e-rara-88566
info.json saved as info/zhb_10_3931_e-rara-88566.json
Record with ID 995285200105505 saved as metadata/zhb_10_3931_e-rara-88566.xml
---
fulldump written to fulldump/erara_complete_set.json
Saved fulldump in excel file as fulldump/erara_complete_set.xlsx
